In [ ]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)

In [ ]:
# ==========================================================
# Cell 2 - Setup
# ==========================================================

import os
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from google import genai

# ----------------------------------------------------------
# Load Environment Variables
# ----------------------------------------------------------

load_dotenv(override=True)

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found!\n"
        "Create a .env file containing:\n"
        "GEMINI_API_KEY=YOUR_API_KEY"
    )

print("Gemini API Key Loaded")

# ----------------------------------------------------------
# Gemini Client
# ----------------------------------------------------------

client = genai.Client(api_key=API_KEY)

# ----------------------------------------------------------
# Show Available Models
# ----------------------------------------------------------

print("\nAvailable Gemini Models:\n")

available_models = []

for model in client.models.list():
    available_models.append(model.name)
    print(model.name)

print("\n")

# ----------------------------------------------------------
# Select Model
# ----------------------------------------------------------

PREFERRED_MODELS = [
    "models/gemini-3.6-flash",
    "models/gemini-3.6-flash-lite",
    "models/gemini-2.5-flash"
]

MODEL_NAME = None

for model in PREFERRED_MODELS:
    if model in available_models:
        MODEL_NAME = model
        break

if MODEL_NAME is None:
    raise ValueError(
        "No supported Gemini model found.\n"
        "Use one of the models printed above."
    )

print("Using Model:", MODEL_NAME)

# ----------------------------------------------------------
# Project Directories
# ----------------------------------------------------------

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
MEMORY_DIR = BASE_DIR / "memory"
CHROMA_DIR = BASE_DIR / "chroma_db"

DATA_DIR.mkdir(exist_ok=True)
MEMORY_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)

print("Project folders ready")

# ----------------------------------------------------------
# Embedding Model
# ----------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(" Embedding model loaded")

# ----------------------------------------------------------
# Persistent ChromaDB
# ----------------------------------------------------------

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

COLLECTION_NAME = "meeting_docs"

try:
    collection = chroma_client.get_collection(COLLECTION_NAME)
    print("Existing Chroma collection loaded")
except:
    collection = chroma_client.create_collection(COLLECTION_NAME)
    print("New Chroma collection created")

print("\n Setup Complete!")

In [ ]:
# ==========================================================
# Cell 3 - Create Sample Dataset & Build RAG Index
# ==========================================================

from uuid import uuid4

# ----------------------------------------------------------
# Sample Documents
# ----------------------------------------------------------

sample_docs = {
    "client_profile.txt": """
Company: Acme Corp

Industry:
Manufacturing

Headquarters:
New York

Primary Contact:
Sarah Johnson
Director of Operations

Current Products:
Inventory Management Platform

Pain Points:
- Manual workflows
- Slow reporting
- ERP integration
- Dashboard customization

Deal Stage:
Proposal Sent

Budget:
$120,000
""",

    "meeting_notes.txt": """
Meeting Date:
15 March 2025

Attendees:
Sarah Johnson
John Miller

Discussion:

Discussed workflow automation.

Client requested:

ERP Integration

Dashboard Customization

SAP Connector

Demo scheduled next week.

Customer looked interested.
""",

    "emails.txt": """
Email Summary

Sarah requested:

Updated pricing

Implementation timeline

SAP compatibility

Technical workshop

Proposal before Friday.
""",

    "action_items.txt": """
Open Action Items

1. Send updated pricing.

2. Prepare ERP demo.

3. Share implementation timeline.

4. Confirm SAP support.

5. Schedule technical workshop.
""",

    "contract.txt": """
Contract

Value:
$120,000

Duration:
2 Years

Status:
Proposal Sent

Decision Expected:
Next Month
"""
}

# ----------------------------------------------------------
# Write Documents
# ----------------------------------------------------------

for filename, text in sample_docs.items():

    (DATA_DIR / filename).write_text(
        text.strip(),
        encoding="utf-8"
    )

print(" Sample documents created")

# ----------------------------------------------------------
# Long-Term Memory
# ----------------------------------------------------------

memory = {
    "Acme Corp": {
        "Relationship": "Warm Lead",
        "Preferred Topics": [
            "ERP Integration",
            "Automation",
            "SAP"
        ],
        "Last Meeting":
            "Client showed strong interest in SAP integration."
    }
}

with open(
    MEMORY_DIR / "long_term_memory.json",
    "w"
) as f:

    json.dump(memory, f, indent=4)

print("Long-term memory created")

# ----------------------------------------------------------
# Chunking Function
# ----------------------------------------------------------

def chunk_text(text, chunk_size=300):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunks.append(
            text[i:i+chunk_size]
        )

    return chunks

# ----------------------------------------------------------
# Remove Existing Documents
# ----------------------------------------------------------

try:

    old = collection.get()

    if old["ids"]:

        collection.delete(ids=old["ids"])

except:
    pass

# ----------------------------------------------------------
# Index into ChromaDB
# ----------------------------------------------------------

count = 0

for file in DATA_DIR.glob("*.txt"):

    text = file.read_text()

    chunks = chunk_text(text)

    for chunk in chunks:

        embedding = embedding_model.encode(
            chunk
        ).tolist()

        collection.add(

            ids=[str(uuid4())],

            documents=[chunk],

            embeddings=[embedding],

            metadatas=[
                {
                    "source": file.name
                }
            ]
        )

        count += 1

print(f"Indexed {count} chunks into ChromaDB")

# ----------------------------------------------------------
# Verify
# ----------------------------------------------------------

print("\nFiles Indexed:\n")

for file in DATA_DIR.glob("*.txt"):

    print("•", file.name)

In [ ]:
# ==========================================================
# Cell 4 - Agent Tools
# ==========================================================

import json

# ----------------------------------------------------------
# Tool Class
# ----------------------------------------------------------

class AgentTools:

    def __init__(self, collection, embedding_model):

        self.collection = collection
        self.embedding_model = embedding_model

    # ------------------------------------------------------
    # Tool 1 : RAG Search
    # ------------------------------------------------------

    def search_documents(self, query, top_k=3):

        query_embedding = self.embedding_model.encode(
            query
        ).tolist()

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        retrieved = []

        if results["documents"]:

            for doc, meta in zip(
                results["documents"][0],
                results["metadatas"][0]
            ):

                retrieved.append({
                    "source": meta["source"],
                    "content": doc
                })

        return retrieved

    # ------------------------------------------------------
    # Tool 2 : Meeting Notes
    # ------------------------------------------------------

    def get_meeting_notes(self):

        path = DATA_DIR / "meeting_notes.txt"

        if path.exists():
            return path.read_text()

        return ""

    # ------------------------------------------------------
    # Tool 3 : Email Summary
    # ------------------------------------------------------

    def get_emails(self):

        path = DATA_DIR / "emails.txt"

        if path.exists():
            return path.read_text()

        return ""

    # ------------------------------------------------------
    # Tool 4 : Action Items
    # ------------------------------------------------------

    def get_action_items(self):

        path = DATA_DIR / "action_items.txt"

        if path.exists():
            return path.read_text()

        return ""

    # ------------------------------------------------------
    # Tool 5 : Long-Term Memory
    # ------------------------------------------------------

    def get_long_term_memory(self, client_name):

        path = MEMORY_DIR / "long_term_memory.json"

        if not path.exists():
            return {}

        with open(path, "r") as f:

            memory = json.load(f)

        return memory.get(client_name, {})

# ----------------------------------------------------------
# Initialize Tools
# ----------------------------------------------------------

tools = AgentTools(
    collection=collection,
    embedding_model=embedding_model
)

print(" Agent Tools Initialized")

In [ ]:
docs = tools.search_documents("Acme Corp ERP Integration")

for d in docs:
    print("=" * 40)
    print("Source:", d["source"])
    print(d["content"])

In [ ]:
print(tools.get_meeting_notes())

In [ ]:
print(tools.get_emails())

In [ ]:
print(tools.get_action_items())

In [ ]:
print(
    tools.get_long_term_memory("Acme Corp")
)

In [ ]:
# ==========================================================
# Cell 5 - Meeting Preparation Agent
# ==========================================================

import json

# ----------------------------------------------------------
# Short-Term Memory
# ----------------------------------------------------------

class ShortTermMemory:

    def __init__(self):
        self.history = []

    def add(self, role, message):
        self.history.append({
            "role": role,
            "message": message
        })

        # Keep last 10 messages
        self.history = self.history[-10:]

    def get_context(self):
        if not self.history:
            return "No previous conversation."

        return "\n".join(
            [f"{h['role']}: {h['message']}" for h in self.history]
        )


# ----------------------------------------------------------
# Meeting Agent
# ----------------------------------------------------------

class MeetingPreparationAgent:

    def __init__(self, client, model_name, tools):

        self.client = client
        self.model_name = model_name
        self.tools = tools
        self.memory = ShortTermMemory()

    # ------------------------------------------------------

    def plan(self, query):

        q = query.lower()

        plan = []

        if "meeting" in q or "client" in q:
            plan = [
                "search_documents",
                "meeting_notes",
                "emails",
                "action_items",
                "long_term_memory"
            ]

        elif "task" in q or "action" in q:
            plan = [
                "action_items"
            ]

        elif "email" in q:
            plan = [
                "emails"
            ]

        elif "note" in q:
            plan = [
                "meeting_notes"
            ]

        else:
            plan = [
                "search_documents"
            ]

        return plan

    # ------------------------------------------------------

    def run(self, query):

        print("=" * 60)
        print("🤖 AI Agent")
        print("=" * 60)

        print("Planning...")

        tools_to_use = self.plan(query)

        print("Plan:", tools_to_use)

        context = ""

        # ------------------------------
        # Tool Execution
        # ------------------------------

        if "search_documents" in tools_to_use:

            docs = self.tools.search_documents(query)

            context += "\nRelevant Documents\n\n"

            for d in docs:

                context += f"""
Source:
{d['source']}

{d['content']}

"""

        if "meeting_notes" in tools_to_use:

            context += "\nMeeting Notes\n\n"

            context += self.tools.get_meeting_notes()

            context += "\n"

        if "emails" in tools_to_use:

            context += "\nEmails\n\n"

            context += self.tools.get_emails()

            context += "\n"

        if "action_items" in tools_to_use:

            context += "\nAction Items\n\n"

            context += self.tools.get_action_items()

            context += "\n"

        if "long_term_memory" in tools_to_use:

            memory = self.tools.get_long_term_memory(
                "Acme Corp"
            )

            context += "\nLong Term Memory\n\n"

            context += json.dumps(
                memory,
                indent=2
            )

        # ------------------------------
        # Prompt
        # ------------------------------

        prompt = f"""
You are an intelligent Meeting Preparation AI Agent.

Answer the user's question using ONLY the retrieved information below.

If the answer is not available, clearly say so.

Conversation History:

{self.memory.get_context()}

Retrieved Context:

{context}

User Question:

{query}
"""

        print("Calling Gemini...")

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt
        )

        self.memory.add("User", query)

        self.memory.add("Assistant", response.text)

        return response.text


# ----------------------------------------------------------
# Initialize Agent
# ----------------------------------------------------------

agent = MeetingPreparationAgent(
    client,
    MODEL_NAME,
    tools
)

print(" Meeting Agent Ready")

In [ ]:
# ==========================================================
# Cell 6 - Interactive Chat
# ==========================================================

while True:

    query = input("\nYou: ")

    if query.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    answer = agent.run(query)

    print("\nAgent:\n")
    print(answer)